# Frontier RCCL-Allreduce — Plots

Adapted from the SULI 2025 notebook. Raw data is OSU / rccl-tests **text output**
(`results/sweep/N<nodes>_job<id>/{A,B,C,D,E}_*.txt`), parsed below into a tidy
DataFrame `data` with columns `[nodes, config, size, avg, std, lo, hi, nreps]`
(averaged over the repetition jobs).

Lives in `frontier/plots/`; open in Jupyter or **VSCode** (Python + Jupyter extensions;
needs `pandas numpy matplotlib`). The data path auto-detects whether the working directory
is `frontier/` or `plots/`, so no manual `cd` is needed. Configs:
A=MPICH host(CPU), B=MPICH dev(CPU algo), C=**MPICH+RCCL**, D=Cray MPICH, E=rccl-tests ceiling.

In [ ]:
# === Load & clean the sweep data =========================================
import os, re, glob
import numpy as np, pandas as pd

RESULTS = "results/sweep" if os.path.isdir("results/sweep") else "../results/sweep"

OSU_FILES = {"A":"A_mpich_host.txt","B":"B_mpich_dev.txt",
             "C":"C_mpich_rccl.txt","D":"D_cray_gpuaware.txt"}
LABEL = {"A":"MPICH host (CPU)","B":"MPICH dev (CPU algo)","C":"MPICH + RCCL",
         "D":"Cray MPICH","E":"RCCL (rccl-tests)"}

def parse_osu(path):
    # OSU collective output -> rows of (size_bytes, avg_us, min_us, max_us)
    out = []
    for line in open(path):
        p = line.split()
        if len(p) >= 2 and p[0].isdigit():
            try:
                out.append((int(p[0]), float(p[1]),
                            float(p[2]) if len(p) > 2 else np.nan,
                            float(p[3]) if len(p) > 3 else np.nan))
            except ValueError:
                pass
    return out

def parse_rccl_tests(path):
    # rccl-tests all_reduce_perf -> (size, in-place time_us, in-place busbw_GBps)
    # columns: size count type redop root  time algbw busbw #wrong  time algbw busbw #wrong
    out = []
    for line in open(path):
        p = line.split()
        if len(p) >= 8 and p[0].isdigit():
            try:
                out.append((int(p[0]), float(p[5]), float(p[7])))
            except (ValueError, IndexError):
                pass
    return out

rows = []
for d in sorted(glob.glob(os.path.join(RESULTS, "N*_job*"))):
    m = re.match(r"N(\d+)_job(\d+)", os.path.basename(d))
    if not m:
        continue
    nodes, job = int(m.group(1)), int(m.group(2))
    for cfg, fn in OSU_FILES.items():
        fp = os.path.join(d, fn)
        if os.path.exists(fp):
            for s, a, mn, mx in parse_osu(fp):
                rows.append(dict(nodes=nodes, job=job, config=cfg, size=s,
                                 avg=a, min=mn, max=mx))
    fe = os.path.join(d, "E_rccl_tests.txt")
    if os.path.exists(fe):
        for s, t, bw in parse_rccl_tests(fe):
            rows.append(dict(nodes=nodes, job=job, config="E", size=s,
                             avg=t, min=np.nan, max=np.nan))

raw = pd.DataFrame(rows)
raw = raw[raw["avg"] > 0]
# average over reps (jobs) -> one row per (nodes, config, size)
data = (raw.groupby(["nodes","config","size"], as_index=False)
           .agg(avg=("avg","mean"), std=("avg","std"),
                lo=("min","mean"), hi=("max","mean"), nreps=("job","nunique")))
print("configs :", sorted(data.config.unique()))
print("nodes   :", sorted(data.nodes.unique()))
print("sizes   :", data["size"].nunique(), "distinct")
data.head()

: 

In [ ]:
# === Plot style (from the SULI 2025 notebook) ============================
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

mpl.rcParams["font.family"] = "STIXGeneral"
mpl.rcParams["mathtext.fontset"] = "stix"
mpl.rcParams.update({"font.size":18,"axes.titlesize":22,"axes.labelsize":20,
                     "legend.fontsize":13,"xtick.labelsize":16,"ytick.labelsize":16})

STYLE = {
    "A": dict(color="#888888", marker="v", linestyle="--", linewidth=2,   label=LABEL["A"]),
    "B": dict(color="blue",    marker="o", linestyle="--", linewidth=2,   label=LABEL["B"]),
    "C": dict(color="red",     marker="s", linestyle="-",  linewidth=3,   markersize=8, label=LABEL["C"]),
    "D": dict(color="#a62ce8", marker="^", linestyle="-",  linewidth=2.5, label=LABEL["D"]),
    "E": dict(color="green",   marker="*", linestyle=":",  linewidth=2,   markersize=12, label=LABEL["E"]),
}
ORDER = ["A","B","C","D","E"]

def sci(x,_):
    if x <= 0: return "0"
    e = int(np.floor(np.log10(x))); b = x/10**e
    return rf"${int(round(b))}\times10^{{{e}}}$"

def human(nb):
    nb=float(nb)
    for u in ["B","KiB","MiB","GiB"]:
        if nb < 1024: return f"{int(nb)}{u}"
        nb/=1024
    return f"{int(nb)}TiB"

def series(nodes, config):
    d = data[(data.nodes==nodes)&(data.config==config)].sort_values("size")
    return d["size"].values, d["avg"].values

In [ ]:
# === Latency vs message size at one node count (all configs) =============
# Right axis: RCCL speedup over a baseline (default Cray, "D"; try "B").
def plot_latency_vs_size(nodes, speedup_over="D", save=None):
    fig, ax1 = plt.subplots(figsize=(12,8))
    for cfg in ORDER:
        x,y = series(nodes,cfg)
        if len(x): ax1.plot(x,y,**STYLE[cfg])
    ax1.set_xscale("log"); ax1.set_yscale("log")
    ax1.set_xlabel("Message size (bytes, log)"); ax1.set_ylabel("Latency (\u00b5s, log)")
    ax1.set_title(f"Allreduce latency \u2014 {nodes} node(s)")
    ax1.yaxis.set_major_formatter(FuncFormatter(sci))
    ax1.grid(True, which="both", ls="--", alpha=0.4)

    xc,yc = series(nodes,"C"); xb,yb = series(nodes,speedup_over)
    if len(xc) and len(xb):
        common = np.intersect1d(xc,xb)
        sc = np.array([yb[list(xb).index(s)]/yc[list(xc).index(s)] for s in common])
        ax2 = ax1.twinx()
        ax2.plot(common, sc, marker="D", color="#04c939", lw=3, label=f"Speedup C/{speedup_over}")
        ax2.axhline(1, color="black", ls="--", lw=1)
        ax2.set_yscale("log")
        ax2.set_ylabel(f"Speedup (C / {speedup_over})", rotation=270, labelpad=25)
        ax2.yaxis.set_major_formatter(FuncFormatter(lambda y,_: f"{y:.0f}"))
        i = int(np.argmax(sc))
        ax2.annotate(f"{sc[i]:.1f}\u00d7", xy=(common[i],sc[i]),
                     xytext=(common[i], sc[i]*0.6), fontsize=16,
                     bbox=dict(fc="white",ec="black",boxstyle="round,pad=0.3"),
                     arrowprops=dict(arrowstyle="->"))
        h1,l1 = ax1.get_legend_handles_labels(); h2,l2 = ax2.get_legend_handles_labels()
        ax1.legend(h1+h2, l1+l2, loc="upper left", framealpha=1)
    else:
        ax1.legend(loc="upper left", framealpha=1)
    fig.patch.set_facecolor("white"); plt.tight_layout()
    if save: plt.savefig(save, facecolor="white")
    plt.show()

plot_latency_vs_size(1)          # <- change node count: 1,2,4,8,16,32,64,...

In [ ]:
# === RCCL-vs-baseline speedup vs size, one curve per node count ==========
# Shows the crossover sliding with scale (curve crosses y=1 at smaller sizes
# as node count grows).
def plot_speedup_vs_size(baseline="D", save=None):
    fig, ax = plt.subplots(figsize=(12,8))
    for n in sorted(data.nodes.unique()):
        xc,yc = series(n,"C"); xb,yb = series(n,baseline)
        common = np.intersect1d(xc,xb)
        if not len(common): continue
        sc = np.array([yb[list(xb).index(s)]/yc[list(xc).index(s)] for s in common])
        ax.plot(common, sc, marker="o", lw=2, label=f"{n} nodes")
    ax.axhline(1, color="black", ls="--", lw=1.5)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Message size (bytes, log)")
    ax.set_ylabel(f"Speedup: {LABEL['C']} / {LABEL[baseline]}")
    ax.set_title(f"RCCL speedup over {LABEL[baseline]} (per node count)")
    ax.grid(True, which="both", ls="--", alpha=0.4)
    ax.legend(title="scale", framealpha=1, ncol=2)
    fig.patch.set_facecolor("white"); plt.tight_layout()
    if save: plt.savefig(save, facecolor="white")
    plt.show()

plot_speedup_vs_size("D")         # baseline "D"=Cray, or "B"=MPICH-CPU

In [ ]:
# === Scaling: latency vs node count at a fixed message size ==============
def plot_scaling(size, save=None):
    fig, ax = plt.subplots(figsize=(12,8))
    for cfg in ORDER:
        d = data[(data.config==cfg)&(data["size"]==size)].sort_values("nodes")
        if len(d): ax.plot(d.nodes, d.avg, **STYLE[cfg])
    ax.set_xscale("log", base=2); ax.set_yscale("log")
    ax.set_xlabel("Nodes (log2)"); ax.set_ylabel("Latency (\u00b5s, log)")
    ax.set_title(f"Allreduce scaling @ {human(size)}")
    ax.yaxis.set_major_formatter(FuncFormatter(sci))
    ax.grid(True, which="both", ls="--", alpha=0.4); ax.legend(framealpha=1)
    fig.patch.set_facecolor("white"); plt.tight_layout()
    if save: plt.savefig(save, facecolor="white")
    plt.show()

plot_scaling(1048576)             # 1 MiB; try 16777216 (16MiB), 1073741824 (1GiB)

In [ ]:
# === Crossover surface: who wins (C vs baseline) over size x nodes =======
# color = log2(baseline/C): >0 (red) RCCL faster, <0 (blue) baseline faster.
def plot_crossover(baseline="D", save=None):
    pc = data[data.config=="C"].pivot_table(index="size", columns="nodes", values="avg")
    pb = data[data.config==baseline].pivot_table(index="size", columns="nodes", values="avg")
    cols = sorted(set(pc.columns) & set(pb.columns))
    idx  = sorted(set(pc.index) & set(pb.index))
    ratio = np.log2(pb.loc[idx,cols] / pc.loc[idx,cols])
    fig, ax = plt.subplots(figsize=(12,8))
    vmax = np.nanmax(np.abs(ratio.values))
    im = ax.imshow(ratio.values, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                   origin="lower", aspect="auto")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols)
    ax.set_yticks(range(len(idx)));  ax.set_yticklabels([human(s) for s in idx])
    ax.set_xlabel("Nodes"); ax.set_ylabel("Message size")
    ax.set_title(f"log2({LABEL[baseline]} / {LABEL['C']})  \u2014  red = RCCL faster")
    cb = fig.colorbar(im, ax=ax); cb.set_label(f"log2 speedup (C vs {baseline})")
    fig.patch.set_facecolor("white"); plt.tight_layout()
    if save: plt.savefig(save, facecolor="white")
    plt.show()

plot_crossover("D")